# 04 - MLP (AI Infra 视角)

本节从 **工程实现** 角度理解 MLP：
- 参数量与显存分析
- 激活函数的计算特性
- SwiGLU vs 标准 MLP
- Kernel Fusion 优化

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. MLP 核心 (30秒版)

```
x (dim) → Linear → 激活 → Linear → 输出 (dim)
          升维 4x            降维 1/4

参数量: 2 × dim × (4 × dim) = 8 × dim²
占模型总参数: ~2/3 (比 Attention 多!)
```

## 2. 参数量分析 (重要!)

MLP 是 Transformer 中参数最多的部分:

```
每层参数:
  Attention: 4 × dim² (Q, K, V, O)
  MLP:       8 × dim² (up + down)
  
MLP 占比: 8/(4+8) = 67%
```

In [2]:
def mlp_params(dim, expansion=4):
    """计算 MLP 参数量"""
    up = dim * (expansion * dim)    # 升维
    down = (expansion * dim) * dim  # 降维
    return up + down

def attention_params(dim):
    """计算 Attention 参数量 (无 bias)"""
    return 4 * dim * dim  # Q, K, V, O

# 不同模型规模
print("参数量分布:")
for dim in [768, 2048, 4096, 8192]:
    attn = attention_params(dim)
    mlp = mlp_params(dim)
    total = attn + mlp
    print(f"  dim={dim:4d}: Attn {attn/1e6:.1f}M ({attn/total*100:.0f}%), MLP {mlp/1e6:.1f}M ({mlp/total*100:.0f}%)")

参数量分布:
  dim= 768: Attn 2.4M (33%), MLP 4.7M (67%)
  dim=2048: Attn 16.8M (33%), MLP 33.6M (67%)
  dim=4096: Attn 67.1M (33%), MLP 134.2M (67%)
  dim=8192: Attn 268.4M (33%), MLP 536.9M (67%)


## 3. 显存分析

训练时 MLP 的显存占用:

```
激活值 (需要保存用于反向传播):
  输入 x:      B × T × dim
  升维后 h:    B × T × 4×dim  ← 最大!
  激活后:      B × T × 4×dim

总激活显存 ≈ 2 × B × T × 4 × dim × dtype_bytes
```

In [3]:
def mlp_activation_memory(batch, seq_len, dim, expansion=4, dtype_bytes=2):
    """MLP 激活值显存 (BF16)"""
    # 需要保存: 输入 x, 升维后 h (用于激活函数反向)
    h_size = batch * seq_len * expansion * dim * dtype_bytes
    return h_size / 1e9  # GB

# 7B 模型配置
batch, seq_len, dim, n_layers = 32, 4096, 4096, 32

per_layer = mlp_activation_memory(batch, seq_len, dim)
total = per_layer * n_layers

print(f"MLP 激活显存 (batch={batch}, seq={seq_len}):")
print(f"  每层: {per_layer:.2f} GB")
print(f"  全部: {total:.1f} GB")

MLP 激活显存 (batch=32, seq=4096):
  每层: 4.29 GB
  全部: 137.4 GB


## 4. 激活函数对比

| 激活函数 | 计算量 | 显存 | 使用模型 |
|---------|--------|------|----------|
| ReLU | 1x | 最少 | 经典 CNN |
| GELU | ~10x | 同上 | GPT-2, BERT |
| SiLU | ~5x | 同上 | LLaMA 1 |
| **ReLU²** | **2x** | **同上** | **nanochat** |

In [5]:
# nanochat 的 MLP
class MLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.c_fc = nn.Linear(dim, 4 * dim, bias=False)
        self.c_proj = nn.Linear(4 * dim, dim, bias=False)
    
    def forward(self, x):
        x = self.c_fc(x)
        x = F.relu(x).square()  # ReLU²: 简单高效
        x = self.c_proj(x)
        return x

# 测试
mlp = MLP(768)
x = torch.randn(2, 16, 768)
print(f"参数量: {sum(p.numel() for p in mlp.parameters()):,}")

参数量: 4,718,592


## 5. SwiGLU vs 标准 MLP

```
标准 MLP:              SwiGLU (LLaMA):
  x → W1 → act → W2      x → W1 → SiLU ─┐
                         x → W3 ────────┤ × → W2
                                        
参数: 8 × dim²          参数: 12 × dim² (多 50%!)
但 hidden_dim 通常调小
```

In [6]:
# SwiGLU MLP (LLaMA 风格)
class SwiGLUMLP(nn.Module):
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        # LLaMA 用 8/3 倍而不是 4 倍，保持参数量相近
        hidden_dim = hidden_dim or int(4 * dim * 2 / 3)
        
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)  # gate
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)  # down
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)  # up
    
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

# 参数量对比
dim = 4096
print(f"标准 MLP:  {sum(p.numel() for p in MLP(dim).parameters())/1e6:.1f}M")
print(f"SwiGLU:    {sum(p.numel() for p in SwiGLUMLP(dim).parameters())/1e6:.1f}M")

标准 MLP:  134.2M
SwiGLU:    134.2M


## 6. Kernel Fusion

MLP 的优化方向: 融合多个操作减少显存访问

```
未融合:                      融合后:
x → W1 → memory              x → [W1 + ReLU² + W2] → memory
    ↓ 读写显存
memory → ReLU² → memory
    ↓ 读写显存  
memory → W2 → memory

3 次显存访问 → 1 次
```

In [ ]:
# torch.compile 自动融合
mlp = MLP(768)

# 编译优化
# mlp_compiled = torch.compile(mlp)
# 会自动融合 Linear + ReLU² + Linear

print("torch.compile 会自动:")
print("  1. 融合连续的 element-wise 操作")
print("  2. 减少中间张量的显存分配")
print("  3. 优化内存访问模式")

## 7. 面试常见问题

### Q1: MLP 为什么要升维 4 倍?

**答**:
- 增加表达能力: 更高维空间可以学习更复杂的模式
- 经验值: 论文实验表明 4x 是效果和效率的平衡点
- LLaMA 的 SwiGLU 用 8/3 倍，因为多了一个门控分支

---

### Q2: ReLU² 相比 GELU 的优势?

**答**:
- 计算更简单: max(0,x)² vs x·Φ(x)
- 稀疏性: 负值输出为 0，只激活部分神经元
- 平滑: 在 x=0 处导数连续 (ReLU 不连续)
- 效果相当，但更快

---

### Q3: MLP 为什么不用 bias?

**答**:
- 参数节省很小 (约 0.1%)
- 配合 RMSNorm，bias 作用减弱
- 实验表明对效果影响可忽略
- 简化实现

---

### Q4: MLP 和 Attention 在模型中的作用?

**答**:
- **Attention**: 搬运信息，让 token 之间交互
- **MLP**: 存储知识，对每个 token 独立变换
- 研究表明 MLP 存储了事实知识 (可以通过编辑 MLP 权重修改知识)

---

### Q5: SwiGLU 为什么效果更好?

**答**:
- 门控机制: 让模型学习选择哪些特征通过
- 更强的非线性: SiLU × gate 比单一激活函数更灵活
- 代价是多 50% 参数 (通常调小 hidden_dim 补偿)

## 8. 总结速查表

| 主题 | 要点 |
|------|------|
| **参数占比** | MLP 占每层参数的 67% (8dim² vs Attn 4dim²) |
| **升维倍数** | 标准 4x，SwiGLU 用 8/3x |
| **激活函数** | ReLU² 简单高效，SwiGLU 效果好但慢 |
| **显存瓶颈** | 中间激活值 B×T×4×dim |
| **优化** | Kernel Fusion 减少显存访问 |

### nanochat MLP 核心代码

```python
class MLP(nn.Module):
    def __init__(self, dim):
        self.c_fc = nn.Linear(dim, 4 * dim, bias=False)
        self.c_proj = nn.Linear(4 * dim, dim, bias=False)
    
    def forward(self, x):
        x = self.c_fc(x)
        x = F.relu(x).square()  # ReLU²
        x = self.c_proj(x)
        return x
```